In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
import numpy as np
import json
import os
import urllib.request
import bz2

# ============================================================================
# THREE-DOMAIN UNIVERSAL TRANSFER LEARNING
# Domains: MNIST + USPS + SVHN (all digits 0-9, semantically meaningful)
#
# KEY DESIGN: Domain-Specific Batch Normalization
# - Shared conv weights  → learn universal features across all domains
# - Domain-specific BN   → normalize each domain's statistics separately
# - Same Z space         → one universal latent manifold by construction
#
# This directly proves ULHM's claim:
# "A single universal Z space exists that represents all domains
#  and enables transfer because all domains share the same topology."
# ============================================================================

DOMAIN_MNIST = 0
DOMAIN_USPS  = 1
DOMAIN_SVHN  = 2
NUM_DOMAINS  = 3


# ============================================================================
# 1. USPS MANUAL DOWNLOAD
# torchvision USPS URL is broken — use working mirror
# ============================================================================

class USPSDataset(Dataset):
    """
    Manual USPS dataset loader from working mirror.
    Avoids broken torchvision USPS download URL.
    """
    URLS = {
        'train': 'https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/multiclass/usps.bz2',
        'test':  'https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/multiclass/usps.t.bz2'
    }

    def __init__(self, root, train=True, transform=None, download=True):
        self.root = os.path.join(root, 'usps_manual')
        self.train = train
        self.transform = transform
        os.makedirs(self.root, exist_ok=True)

        split = 'train' if train else 'test'
        filepath = os.path.join(self.root, f'usps_{split}.bz2')

        if download and not os.path.exists(filepath):
            print(f'Downloading USPS {split} from working mirror...')
            urllib.request.urlretrieve(self.URLS[split], filepath)
            print(f'✓ Downloaded USPS {split}')

        self.data, self.labels = self._load_libsvm(filepath)

    def _load_libsvm(self, filepath):
        data, labels = [], []
        with bz2.open(filepath, 'rt') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                label = int(parts[0]) % 10
                labels.append(label)
                features = np.zeros(256, dtype=np.float32)
                for feat in parts[1:]:
                    if ':' in feat:
                        idx, val = feat.split(':')
                        features[int(idx) - 1] = float(val)
                img = features.reshape(16, 16)
                img = (img + 1.0) / 2.0  # normalize to [0,1]
                data.append(img)
        return np.array(data, dtype=np.float32), np.array(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.FloatTensor(self.data[idx]).unsqueeze(0)  # [1, 16, 16]
        img = img * 2.0 - 1.0  # normalize to [-1, 1]
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label


# ============================================================================
# 2. PREPROCESSING
# All domains → 32×32 × 3 channels
# MNIST/USPS: pad to 32×32 then repeat channel 3 times
# SVHN: already 32×32×3
# ============================================================================

class PadAndRepeatChannels:
    """
    Grayscale [1, H, W] → RGB-like [3, 32, 32]
    Step 1: Pad to 32×32 (center padding)
    Step 2: Repeat channel 3 times
    Preserves all information — no downsampling
    """
    def __call__(self, x):
        c, h, w = x.shape
        pad_h  = (32 - h) // 2
        pad_w  = (32 - w) // 2
        pad_h2 = 32 - h - pad_h
        pad_w2 = 32 - w - pad_w
        x = F.pad(x, [pad_w, pad_w2, pad_h, pad_h2], value=0)
        x = x.repeat(3, 1, 1)  # [1,32,32] → [3,32,32]
        return x


# ============================================================================
# 3. SPARSE MULTI-DOMAIN DATASET
# ============================================================================

class SparseMultiDomainDataset(Dataset):
    """
    Sparse dataset for any domain.
    All images: 32×32×3 after transforms.
    Spatial mask applied consistently across channels.
    domain_id: 0=MNIST, 1=USPS, 2=SVHN
    """
    def __init__(self, base_dataset, sparsity_level=0.15,
                 domain_id=0, domain_name='Unknown'):
        self.base_dataset  = base_dataset
        self.sparsity_level = sparsity_level
        self.domain_id     = domain_id
        self.domain_name   = domain_name
        print(f"{domain_name} Sparse Dataset — Size: {len(base_dataset)}, "
              f"Sparsity: {sparsity_level:.0%} visible")

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        full_image, label = self.base_dataset[idx]  # [3, 32, 32]

        # Same spatial mask for all 3 channels
        spatial_mask = (torch.rand(1, 32, 32) < self.sparsity_level).float()
        mask         = spatial_mask.repeat(3, 1, 1)  # [3, 32, 32]
        sparse_image = full_image * mask

        label = int(label) % 10  # ensure 0-9

        x = {'sparse_image': sparse_image, 'mask': mask}
        s = {'full_image': full_image, 'label': label, 'domain': self.domain_id}
        return x, s


# ============================================================================
# 4. DOMAIN-SPECIFIC BATCH NORMALIZATION RESIDUAL BLOCK
#
# Core design:
#   - Shared conv weights  → same feature transformation for all domains
#   - Domain-specific BN   → separate running stats per domain
#
# Why this works:
#   MNIST, USPS, SVHN have very different pixel statistics.
#   Shared BN would average them → unstable training.
#   Domain-specific BN normalizes each domain independently
#   while conv weights learn universal features.
#
# This is the same design used in:
#   - Google's multilingual translation models
#   - Universal style transfer networks
#   - Multi-domain image recognition (Jin et al., 2018)
# ============================================================================

class ResidualBlockDomainBN(nn.Module):
    """
    Residual block with domain-specific BatchNorm.
    Shared conv weights + domain-specific BN layers.
    forward(x, domain_id) — domain_id selects which BN to use.
    """
    def __init__(self, channels, num_domains=NUM_DOMAINS):
        super().__init__()

        # Shared conv weights — same for all domains
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)

        # Domain-specific BN — one set per domain
        self.bn1 = nn.ModuleList(
            [nn.BatchNorm2d(channels) for _ in range(num_domains)]
        )
        self.bn2 = nn.ModuleList(
            [nn.BatchNorm2d(channels) for _ in range(num_domains)]
        )

    def forward(self, x, domain_id):
        residual = x
        out = F.relu(self.bn1[domain_id](x))
        out = self.conv1(out)
        out = F.relu(self.bn2[domain_id](out))
        out = self.conv2(out)
        return out + residual


# ============================================================================
# 5. SHARED UNIVERSAL ENCODER WITH DOMAIN-SPECIFIC BN
#
# Architecture:
#   Input:  [B, 6, 32, 32]  (3ch sparse image + 3ch mask)
#   Scale1: 32ch residual blocks  (full resolution 32×32)
#   Scale2: 64ch residual blocks  (16×16 after stride-2 conv)
#   Scale3: 128ch residual blocks (8×8)
#   Scale4: 256ch residual blocks (4×4)
#   Output: [B, latent_channels, 4, 4]  → same Z space for all domains
#
# Key property:
#   ALL domains map to the SAME Z space.
#   Z space IS the universal latent manifold M_z.
#   No post-hoc alignment needed — homeomorphism by construction.
# ============================================================================

class SharedUniversalEncoder(nn.Module):
    """
    ONE shared encoder for ALL three domains.
    Shared conv weights + domain-specific BatchNorm.
    All domains map to the same Z space — universal manifold by construction.
    """
    def __init__(self, latent_channels=128, num_res_blocks=3,
                 num_domains=NUM_DOMAINS):
        super().__init__()
        self.latent_channels = latent_channels
        self.num_domains = num_domains

        # ---- Input conv: 6 → 32 ----
        self.input_conv = nn.Conv2d(6, 32, 3, padding=1, bias=False)
        self.input_bn   = nn.ModuleList(
            [nn.BatchNorm2d(32) for _ in range(num_domains)]
        )

        # ---- Scale 1: 32ch, 32×32 ----
        self.res1 = nn.ModuleList(
            [ResidualBlockDomainBN(32, num_domains) for _ in range(num_res_blocks)]
        )

        # ---- Downsample 32→64, 32×32→16×16 ----
        self.down1    = nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False)
        self.down1_bn = nn.ModuleList(
            [nn.BatchNorm2d(64) for _ in range(num_domains)]
        )

        # ---- Scale 2: 64ch, 16×16 ----
        self.res2 = nn.ModuleList(
            [ResidualBlockDomainBN(64, num_domains) for _ in range(num_res_blocks)]
        )

        # ---- Downsample 64→128, 16×16→8×8 ----
        self.down2    = nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False)
        self.down2_bn = nn.ModuleList(
            [nn.BatchNorm2d(128) for _ in range(num_domains)]
        )

        # ---- Scale 3: 128ch, 8×8 ----
        self.res3 = nn.ModuleList(
            [ResidualBlockDomainBN(128, num_domains) for _ in range(num_res_blocks)]
        )

        # ---- Downsample 128→256, 8×8→4×4 ----
        self.down3    = nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False)
        self.down3_bn = nn.ModuleList(
            [nn.BatchNorm2d(256) for _ in range(num_domains)]
        )

        # ---- Scale 4: 256ch, 4×4 ----
        self.res4 = nn.ModuleList(
            [ResidualBlockDomainBN(256, num_domains) for _ in range(num_res_blocks)]
        )

        # ---- Bottleneck: 256→latent_channels ----
        self.bottleneck    = nn.Conv2d(256, latent_channels, 1, bias=False)
        self.bottleneck_bn = nn.ModuleList(
            [nn.BatchNorm2d(latent_channels) for _ in range(num_domains)]
        )

        # ---- Bottleneck residual blocks ----
        self.res_bottleneck = nn.ModuleList(
            [ResidualBlockDomainBN(latent_channels, num_domains)
             for _ in range(num_res_blocks)]
        )

    def forward(self, sparse_image, mask, domain_id):
        """
        Args:
            sparse_image: [B, 3, 32, 32]
            mask:         [B, 3, 32, 32]
            domain_id:    int — 0=MNIST, 1=USPS, 2=SVHN
        Returns:
            z: [B, latent_channels, 4, 4] — universal latent code
        """
        # Concatenate sparse image and mask
        x = torch.cat([sparse_image, mask], dim=1)  # [B, 6, 32, 32]

        # Input conv + domain BN
        x = F.relu(self.input_bn[domain_id](self.input_conv(x)))

        # Scale 1
        for block in self.res1:
            x = block(x, domain_id)

        # Downsample 1
        x = F.relu(self.down1_bn[domain_id](self.down1(x)))

        # Scale 2
        for block in self.res2:
            x = block(x, domain_id)

        # Downsample 2
        x = F.relu(self.down2_bn[domain_id](self.down2(x)))

        # Scale 3
        for block in self.res3:
            x = block(x, domain_id)

        # Downsample 3
        x = F.relu(self.down3_bn[domain_id](self.down3(x)))

        # Scale 4
        for block in self.res4:
            x = block(x, domain_id)

        # Bottleneck
        x = F.relu(self.bottleneck_bn[domain_id](self.bottleneck(x)))

        # Bottleneck residual
        for block in self.res_bottleneck:
            x = block(x, domain_id)

        return x  # [B, latent_channels, 4, 4] — same Z for ALL domains


# ============================================================================
# 6. DOMAIN-SPECIFIC DECODER
# Each domain reconstructs its own images from universal Z
# Input:  [B, latent_channels, 4, 4]
# Output: [B, 3, 32, 32]
# ============================================================================

class ResidualBlock(nn.Module):
    """Standard residual block for decoders (no domain BN needed)"""
    def __init__(self, channels):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(x))
        out = self.conv1(out)
        out = F.relu(self.bn2(out))
        out = self.conv2(out)
        return out + residual


class DomainSpecificDecoder(nn.Module):
    """
    Domain-specific decoder.
    Takes universal Z and reconstructs domain-specific images.
    Standard BN is fine here — each decoder only sees one domain.
    Output: [B, 3, 32, 32]
    """
    def __init__(self, latent_channels=128, num_res_blocks=2):
        super().__init__()

        self.res_bottleneck = nn.Sequential(
            *[ResidualBlock(latent_channels) for _ in range(num_res_blocks)]
        )
        # 4×4 → 8×8
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 256, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU()
        )
        self.res1 = nn.Sequential(*[ResidualBlock(256) for _ in range(num_res_blocks)])

        # 8×8 → 16×16
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU()
        )
        self.res2 = nn.Sequential(*[ResidualBlock(128) for _ in range(num_res_blocks)])

        # 16×16 → 32×32
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.res3 = nn.Sequential(*[ResidualBlock(64) for _ in range(num_res_blocks)])

        # Output: 3 channels
        self.output_conv = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 3,  3, padding=1), nn.Tanh()
        )

    def forward(self, z):
        x = self.res_bottleneck(z)
        x = self.up1(x)
        x = self.res1(x)
        x = self.up2(x)
        x = self.res2(x)
        x = self.up3(x)
        x = self.res3(x)
        return self.output_conv(x)


# ============================================================================
# 7. DOMAIN-INVARIANT PROJECTION NETWORK
# One per domain — fine-tunes Z for cross-domain alignment
# ============================================================================

class DomainInvariantProjection(nn.Module):
    def __init__(self, latent_channels=128, num_res_blocks=2):
        super().__init__()
        self.projection = nn.Sequential(
            *[ResidualBlock(latent_channels) for _ in range(num_res_blocks)],
            nn.Conv2d(latent_channels, latent_channels, 1),
            nn.BatchNorm2d(latent_channels)
        )

    def forward(self, z):
        return self.projection(z)


# ============================================================================
# 8. SPATIAL CLASSIFIER
# ============================================================================

class SpatialClassifier(nn.Module):
    def __init__(self, latent_channels=128, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(latent_channels, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout2d(0.3),
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(), nn.Dropout2d(0.3),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(512, num_classes)

    def forward(self, z):
        f = self.features(z)
        f = f.view(f.size(0), -1)
        return self.fc(f)


# ============================================================================
# 9. ALIGNMENT LOSSES (three-way)
# ============================================================================

def contrastive_alignment_loss(z_a, z_b, labels_a, labels_b, temperature=0.1):
    """Contrastive alignment between two domains"""
    z_a_flat = F.normalize(z_a.view(z_a.size(0), -1), p=2, dim=1)
    z_b_flat = F.normalize(z_b.view(z_b.size(0), -1), p=2, dim=1)

    sim = torch.matmul(z_a_flat, z_b_flat.t()) / temperature
    align_mask = (labels_a.unsqueeze(1) == labels_b.unsqueeze(0)).float()

    loss = 0.0
    count = 0
    for i in range(z_a.size(0)):
        pos = align_mask[i]
        if pos.sum() == 0:
            continue
        pos_sim = sim[i][pos.bool()]
        neg_sim = sim[i][(1 - pos).bool()]
        pos_exp = torch.exp(pos_sim)
        neg_exp = torch.exp(neg_sim).sum()
        loss += -torch.log(pos_exp / (pos_exp + neg_exp + 1e-8)).mean()
        count += 1

    return loss / count if count > 0 else torch.tensor(0.0, device=z_a.device)


def class_centroid_alignment_loss(z_a, z_b, labels_a, labels_b, num_classes=10):
    """Centroid alignment between two domains"""
    z_a_flat = z_a.view(z_a.size(0), -1)
    z_b_flat = z_b.view(z_b.size(0), -1)

    loss = 0.0
    count = 0
    for c in range(num_classes):
        mask_a = (labels_a == c)
        mask_b = (labels_b == c)
        if mask_a.sum() == 0 or mask_b.sum() == 0:
            continue
        loss += F.mse_loss(z_a_flat[mask_a].mean(0), z_b_flat[mask_b].mean(0))
        count += 1

    return loss / count if count > 0 else torch.tensor(0.0, device=z_a.device)


def three_way_alignment_loss(z_mnist, z_usps, z_svhn,
                              labels_mnist, labels_usps, labels_svhn,
                              temperature=0.1):
    """
    Three-way alignment across all domain pairs simultaneously.
    Pairs: MNIST↔USPS, MNIST↔SVHN, USPS↔SVHN
    """
    # Contrastive losses
    lc_mu = contrastive_alignment_loss(z_mnist, z_usps,  labels_mnist, labels_usps,  temperature)
    lc_ms = contrastive_alignment_loss(z_mnist, z_svhn,  labels_mnist, labels_svhn,  temperature)
    lc_us = contrastive_alignment_loss(z_usps,  z_svhn,  labels_usps,  labels_svhn,  temperature)

    # Centroid losses
    lk_mu = class_centroid_alignment_loss(z_mnist, z_usps, labels_mnist, labels_usps)
    lk_ms = class_centroid_alignment_loss(z_mnist, z_svhn, labels_mnist, labels_svhn)
    lk_us = class_centroid_alignment_loss(z_usps,  z_svhn, labels_usps,  labels_svhn)

    loss_contrastive = (lc_mu + lc_ms + lc_us) / 3.0
    loss_centroid    = (lk_mu + lk_ms + lk_us) / 3.0

    return loss_contrastive, loss_centroid


# ============================================================================
# 10. STEP 1: PRE-TRAIN SHARED ENCODER (Domain-Specific BN)
# Train shared encoder jointly on all three domains.
# Each domain uses its own BN stats — shared conv weights learn universal features.
# ============================================================================

def pretrain_shared_encoder(shared_encoder,
                             decoder_mnist, decoder_usps, decoder_svhn,
                             train_loader_mnist, train_loader_usps, train_loader_svhn,
                             val_loader_mnist,   val_loader_usps,   val_loader_svhn,
                             max_epochs=100, patience=15, device='cpu'):
    """
    Stage 1: Joint pre-training of shared encoder on all three domains.

    Each domain passes through the shared encoder with its own domain_id,
    which selects the appropriate domain-specific BN layers.
    Conv weights are updated by gradients from ALL three domains simultaneously.

    This enforces that the same conv weights must work for MNIST, USPS, and SVHN
    — producing universal features in the same Z space.
    """
    print("=" * 70)
    print("STEP 1: JOINT PRE-TRAINING — SHARED ENCODER + DOMAIN-SPECIFIC BN")
    print("Shared conv weights learn universal features across all 3 domains")
    print("Domain-specific BN normalizes each domain's statistics separately")
    print("ALL domains map to the SAME Z space — universal manifold")
    print("=" * 70 + "\n")

    shared_encoder.to(device)
    decoder_mnist.to(device)
    decoder_usps.to(device)
    decoder_svhn.to(device)

    optimizer = torch.optim.Adam(
        list(shared_encoder.parameters()) +
        list(decoder_mnist.parameters()) +
        list(decoder_usps.parameters()) +
        list(decoder_svhn.parameters()),
        lr=1e-3, weight_decay=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    best_val_loss = float('inf')
    best_epoch    = 0
    no_improve    = 0
    best_state    = None

    for epoch in range(max_epochs):
        shared_encoder.train()
        decoder_mnist.train()
        decoder_usps.train()
        decoder_svhn.train()

        total_loss  = 0
        loss_mnist_ = 0
        loss_usps_  = 0
        loss_svhn_  = 0

        mnist_iter = iter(train_loader_mnist)
        usps_iter  = iter(train_loader_usps)
        svhn_iter  = iter(train_loader_svhn)

        num_batches = min(len(train_loader_mnist),
                         len(train_loader_usps),
                         len(train_loader_svhn))

        for batch_idx in range(num_batches):
            try:
                x_mnist, s_mnist = next(mnist_iter)
                x_usps,  s_usps  = next(usps_iter)
                x_svhn,  s_svhn  = next(svhn_iter)
            except StopIteration:
                break

            # MNIST
            sp_mnist   = x_mnist['sparse_image'].to(device)
            mk_mnist   = x_mnist['mask'].to(device)
            full_mnist = s_mnist['full_image'].to(device)

            # USPS
            sp_usps    = x_usps['sparse_image'].to(device)
            mk_usps    = x_usps['mask'].to(device)
            full_usps  = s_usps['full_image'].to(device)

            # SVHN
            sp_svhn    = x_svhn['sparse_image'].to(device)
            mk_svhn    = x_svhn['mask'].to(device)
            full_svhn  = s_svhn['full_image'].to(device)

            optimizer.zero_grad()

            # Each domain uses its own domain_id → own BN stats
            # Same conv weights process all three
            z_mnist = shared_encoder(sp_mnist, mk_mnist, DOMAIN_MNIST)
            z_usps  = shared_encoder(sp_usps,  mk_usps,  DOMAIN_USPS)
            z_svhn  = shared_encoder(sp_svhn,  mk_svhn,  DOMAIN_SVHN)

            # Domain-specific reconstruction
            recon_mnist = decoder_mnist(z_mnist)
            recon_usps  = decoder_usps(z_usps)
            recon_svhn  = decoder_svhn(z_svhn)

            lm = F.mse_loss(recon_mnist, full_mnist)
            lu = F.mse_loss(recon_usps,  full_usps)
            ls = F.mse_loss(recon_svhn,  full_svhn)

            loss = lm + lu + ls
            loss.backward()

            torch.nn.utils.clip_grad_norm_(shared_encoder.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(decoder_mnist.parameters(),  1.0)
            torch.nn.utils.clip_grad_norm_(decoder_usps.parameters(),   1.0)
            torch.nn.utils.clip_grad_norm_(decoder_svhn.parameters(),   1.0)
            optimizer.step()

            total_loss  += loss.item()
            loss_mnist_ += lm.item()
            loss_usps_  += lu.item()
            loss_svhn_  += ls.item()

            if batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}: '
                      f'Total={loss.item():.4f} '
                      f'(MNIST={lm.item():.4f}, '
                      f'USPS={lu.item():.4f}, '
                      f'SVHN={ls.item():.4f})')

        # Validation
        shared_encoder.eval()
        decoder_mnist.eval()
        decoder_usps.eval()
        decoder_svhn.eval()

        val_total = 0
        val_m = 0
        val_u = 0
        val_s = 0

        with torch.no_grad():
            mv_iter = iter(val_loader_mnist)
            uv_iter = iter(val_loader_usps)
            sv_iter = iter(val_loader_svhn)
            nv = min(len(val_loader_mnist),
                     len(val_loader_usps),
                     len(val_loader_svhn))

            for _ in range(nv):
                try:
                    xm, sm = next(mv_iter)
                    xu, su = next(uv_iter)
                    xs, ss = next(sv_iter)
                except StopIteration:
                    break

                zm = shared_encoder(xm['sparse_image'].to(device),
                                    xm['mask'].to(device), DOMAIN_MNIST)
                zu = shared_encoder(xu['sparse_image'].to(device),
                                    xu['mask'].to(device), DOMAIN_USPS)
                zs = shared_encoder(xs['sparse_image'].to(device),
                                    xs['mask'].to(device), DOMAIN_SVHN)

                lm = F.mse_loss(decoder_mnist(zm), sm['full_image'].to(device))
                lu = F.mse_loss(decoder_usps(zu),  su['full_image'].to(device))
                ls = F.mse_loss(decoder_svhn(zs),  ss['full_image'].to(device))

                val_total += (lm + lu + ls).item()
                val_m += lm.item()
                val_u += lu.item()
                val_s += ls.item()

        avg_val = val_total / nv

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != old_lr:
            print(f'  → LR reduced: {old_lr:.2e} → {new_lr:.2e}')

        print(f'\nEpoch {epoch+1}/{max_epochs}:')
        print(f'  TRAIN Total={total_loss/num_batches:.4f} '
              f'(MNIST={loss_mnist_/num_batches:.4f}, '
              f'USPS={loss_usps_/num_batches:.4f}, '
              f'SVHN={loss_svhn_/num_batches:.4f})')
        print(f'  VAL   Total={avg_val:.4f} '
              f'(MNIST={val_m/nv:.4f}, '
              f'USPS={val_u/nv:.4f}, '
              f'SVHN={val_s/nv:.4f})')

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_epoch    = epoch + 1
            no_improve    = 0
            best_state = {
                'shared_encoder': shared_encoder.state_dict(),
                'decoder_mnist':  decoder_mnist.state_dict(),
                'decoder_usps':   decoder_usps.state_dict(),
                'decoder_svhn':   decoder_svhn.state_dict(),
            }
            print(f'  ✓ NEW BEST!')
        else:
            no_improve += 1
            print(f'  No improvement for {no_improve} epoch(s)')
            if no_improve >= patience:
                print(f'\nEARLY STOPPING at epoch {epoch+1}')
                break

    if best_state:
        shared_encoder.load_state_dict(best_state['shared_encoder'])
        decoder_mnist.load_state_dict(best_state['decoder_mnist'])
        decoder_usps.load_state_dict(best_state['decoder_usps'])
        decoder_svhn.load_state_dict(best_state['decoder_svhn'])
        print(f"✓ Restored best model from epoch {best_epoch}\n")

    return shared_encoder, decoder_mnist, decoder_usps, decoder_svhn


# ============================================================================
# 11. STEP 2: THREE-WAY PROJECTION + ALIGNMENT (NO CLASSIFIER)
# Shared encoder FROZEN.
# Each domain gets its own projection network.
# Three-way alignment enforces class-level correspondence across domains.
# ============================================================================

def train_three_way_alignment(shared_encoder,
                               projection_mnist, projection_usps, projection_svhn,
                               decoder_mnist, decoder_usps, decoder_svhn,
                               train_loader_mnist, train_loader_usps, train_loader_svhn,
                               val_loader_mnist,   val_loader_usps,   val_loader_svhn,
                               max_epochs=100, patience=15, device='cpu'):
    """
    Stage 2: Three-way projection and alignment.
    Shared encoder is FROZEN — Z space structure is preserved.
    Projection networks fine-tune alignment across domains.
    NO classifier trained here.
    """
    print("=" * 70)
    print("STEP 2: THREE-WAY PROJECTION + ALIGNMENT (NO CLASSIFIER)")
    print("Frozen shared encoder — Z space preserved")
    print("Pairs aligned: MNIST↔USPS, MNIST↔SVHN, USPS↔SVHN")
    print("=" * 70 + "\n")

    # Freeze shared encoder
    for p in shared_encoder.parameters():
        p.requires_grad = False
    shared_encoder.eval()

    shared_encoder.to(device)
    projection_mnist.to(device)
    projection_usps.to(device)
    projection_svhn.to(device)
    decoder_mnist.to(device)
    decoder_usps.to(device)
    decoder_svhn.to(device)

    optimizer = torch.optim.Adam(
        list(projection_mnist.parameters()) +
        list(projection_usps.parameters())  +
        list(projection_svhn.parameters())  +
        list(decoder_mnist.parameters())    +
        list(decoder_usps.parameters())     +
        list(decoder_svhn.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    lambda_cont = 0.5
    lambda_cent = 0.5

    best_val = float('inf')
    best_epoch = 0
    no_improve = 0
    best_state = None

    for epoch in range(max_epochs):
        projection_mnist.train()
        projection_usps.train()
        projection_svhn.train()
        decoder_mnist.train()
        decoder_usps.train()
        decoder_svhn.train()

        total = recon_t = cont_t = cent_t = 0

        mnist_iter = iter(train_loader_mnist)
        usps_iter  = iter(train_loader_usps)
        svhn_iter  = iter(train_loader_svhn)

        nb = min(len(train_loader_mnist),
                 len(train_loader_usps),
                 len(train_loader_svhn))

        for batch_idx in range(nb):
            try:
                xm, sm = next(mnist_iter)
                xu, su = next(usps_iter)
                xs, ss = next(svhn_iter)
            except StopIteration:
                break

            sp_m = xm['sparse_image'].to(device)
            mk_m = xm['mask'].to(device)
            fl_m = sm['full_image'].to(device)
            lb_m = sm['label'].to(device)

            sp_u = xu['sparse_image'].to(device)
            mk_u = xu['mask'].to(device)
            fl_u = su['full_image'].to(device)
            lb_u = su['label'].to(device)

            sp_s = xs['sparse_image'].to(device)
            mk_s = xs['mask'].to(device)
            fl_s = ss['full_image'].to(device)
            lb_s = ss['label'].to(device)

            optimizer.zero_grad()

            # Encode with frozen shared encoder (domain-specific BN)
            with torch.no_grad():
                z_m = shared_encoder(sp_m, mk_m, DOMAIN_MNIST)
                z_u = shared_encoder(sp_u, mk_u, DOMAIN_USPS)
                z_s = shared_encoder(sp_s, mk_s, DOMAIN_SVHN)

            # Project to aligned universal space
            z_m_univ = projection_mnist(z_m)
            z_u_univ = projection_usps(z_u)
            z_s_univ = projection_svhn(z_s)

            # Reconstruction
            lr = (F.mse_loss(decoder_mnist(z_m_univ), fl_m) +
                  F.mse_loss(decoder_usps(z_u_univ),  fl_u) +
                  F.mse_loss(decoder_svhn(z_s_univ),  fl_s))

            # Three-way alignment
            lc, lk = three_way_alignment_loss(
                z_m_univ, z_u_univ, z_s_univ,
                lb_m, lb_u, lb_s
            )

            loss = lr + lambda_cont * lc + lambda_cent * lk
            loss.backward()

            for proj in [projection_mnist, projection_usps, projection_svhn]:
                torch.nn.utils.clip_grad_norm_(proj.parameters(), 1.0)

            optimizer.step()

            total   += loss.item()
            recon_t += lr.item()
            cont_t  += lc.item()
            cent_t  += lk.item()

            if batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}: '
                      f'Loss={loss.item():.4f} '
                      f'(Recon={lr.item():.4f}, '
                      f'Cont={lc.item():.4f}, '
                      f'Cent={lk.item():.4f})')

        # Validation
        for m in [projection_mnist, projection_usps, projection_svhn,
                  decoder_mnist, decoder_usps, decoder_svhn]:
            m.eval()

        vt = vr = vc = vk = 0

        with torch.no_grad():
            mv = iter(val_loader_mnist)
            uv = iter(val_loader_usps)
            sv = iter(val_loader_svhn)
            nv = min(len(val_loader_mnist),
                     len(val_loader_usps),
                     len(val_loader_svhn))

            for _ in range(nv):
                try:
                    xm, sm = next(mv)
                    xu, su = next(uv)
                    xs, ss = next(sv)
                except StopIteration:
                    break

                zm = projection_mnist(shared_encoder(
                    xm['sparse_image'].to(device), xm['mask'].to(device), DOMAIN_MNIST))
                zu = projection_usps(shared_encoder(
                    xu['sparse_image'].to(device), xu['mask'].to(device), DOMAIN_USPS))
                zs = projection_svhn(shared_encoder(
                    xs['sparse_image'].to(device), xs['mask'].to(device), DOMAIN_SVHN))

                lr = (F.mse_loss(decoder_mnist(zm), sm['full_image'].to(device)) +
                      F.mse_loss(decoder_usps(zu),  su['full_image'].to(device)) +
                      F.mse_loss(decoder_svhn(zs),  ss['full_image'].to(device)))

                lc, lk = three_way_alignment_loss(
                    zm, zu, zs,
                    sm['label'].to(device),
                    su['label'].to(device),
                    ss['label'].to(device)
                )

                bl = lr + lambda_cont * lc + lambda_cent * lk
                vt += bl.item()
                vr += lr.item()
                vc += lc.item()
                vk += lk.item()

        avg_val = vt / nv

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != old_lr:
            print(f'  → LR reduced: {old_lr:.2e} → {new_lr:.2e}')

        print(f'\n{"=" * 70}')
        print(f'Epoch {epoch+1}/{max_epochs}:')
        print(f'  TRAIN Total={total/nb:.4f} '
              f'(Recon={recon_t/nb:.4f}, Cont={cont_t/nb:.4f}, Cent={cent_t/nb:.4f})')
        print(f'  VAL   Total={avg_val:.4f} '
              f'(Recon={vr/nv:.4f}, Cont={vc/nv:.4f}, Cent={vk/nv:.4f})')

        if avg_val < best_val:
            best_val   = avg_val
            best_epoch = epoch + 1
            no_improve = 0
            best_state = {
                'projection_mnist': projection_mnist.state_dict(),
                'projection_usps':  projection_usps.state_dict(),
                'projection_svhn':  projection_svhn.state_dict(),
                'decoder_mnist':    decoder_mnist.state_dict(),
                'decoder_usps':     decoder_usps.state_dict(),
                'decoder_svhn':     decoder_svhn.state_dict(),
            }
            print(f'  ✓ NEW BEST!')
        else:
            no_improve += 1
            print(f'  No improvement for {no_improve} epoch(s)')
            if no_improve >= patience:
                print(f'\nEARLY STOPPING at epoch {epoch+1}')
                break

        print(f'{"=" * 70}\n')

    if best_state:
        projection_mnist.load_state_dict(best_state['projection_mnist'])
        projection_usps.load_state_dict(best_state['projection_usps'])
        projection_svhn.load_state_dict(best_state['projection_svhn'])
        decoder_mnist.load_state_dict(best_state['decoder_mnist'])
        decoder_usps.load_state_dict(best_state['decoder_usps'])
        decoder_svhn.load_state_dict(best_state['decoder_svhn'])
        print(f"✓ Restored best model from epoch {best_epoch}\n")

    return projection_mnist, projection_usps, projection_svhn


# ============================================================================
# 12. STEP 3: TRAIN CLASSIFIER ON SINGLE DOMAIN ONLY
# ============================================================================

def train_classifier_single_domain(shared_encoder, projection, classifier,
                                   train_loader, val_loader, domain_id,
                                   domain_name, max_epochs=50, patience=10,
                                   device='cpu'):
    """
    Train classifier on ONE domain only.
    Shared encoder and projection are FROZEN.
    Classifier never sees other domains during training.
    """
    print("=" * 70)
    print(f"STEP 3: TRAINING CLASSIFIER ON {domain_name} ONLY")
    print(f"Classifier never sees other domains")
    print("=" * 70 + "\n")

    for p in shared_encoder.parameters():
        p.requires_grad = False
    for p in projection.parameters():
        p.requires_grad = False

    shared_encoder.eval()
    projection.eval()
    shared_encoder.to(device)
    projection.to(device)
    classifier.to(device)

    optimizer = torch.optim.Adam(
        classifier.parameters(), lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    best_acc   = 0.0
    best_epoch = 0
    no_improve = 0
    best_state = None

    for epoch in range(max_epochs):
        classifier.train()
        correct = total = 0

        for x, s in train_loader:
            sparse = x['sparse_image'].to(device)
            mask   = x['mask'].to(device)
            labels = s['label'].to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                z = shared_encoder(sparse, mask, domain_id)
                z = projection(z)

            logits = classifier(z)
            loss   = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            _, pred = logits.max(1)
            total   += labels.size(0)
            correct += pred.eq(labels).sum().item()

        train_acc = 100. * correct / total

        classifier.eval()
        val_correct = val_total = 0

        with torch.no_grad():
            for x, s in val_loader:
                sparse = x['sparse_image'].to(device)
                mask   = x['mask'].to(device)
                labels = s['label'].to(device)

                z = shared_encoder(sparse, mask, domain_id)
                z = projection(z)
                logits = classifier(z)

                _, pred = logits.max(1)
                val_total   += labels.size(0)
                val_correct += pred.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_acc)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != old_lr:
            print(f'  → LR reduced: {old_lr:.2e} → {new_lr:.2e}')

        print(f'Epoch {epoch+1}/{max_epochs}: '
              f'Train={train_acc:.2f}%, Val={val_acc:.2f}%')

        if val_acc > best_acc:
            best_acc   = val_acc
            best_epoch = epoch + 1
            no_improve = 0
            best_state = {'classifier': classifier.state_dict()}
            print(f'  ✓ NEW BEST!')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'\nEARLY STOPPING at epoch {epoch+1}')
                break

    if best_state:
        classifier.load_state_dict(best_state['classifier'])
        print(f"✓ Restored best classifier from epoch {best_epoch}\n")

    return classifier


# ============================================================================
# 13. EVALUATION
# ============================================================================

def evaluate_transfer(shared_encoder, projection, classifier,
                      test_loader, domain_id,
                      source_name, target_name, device='cpu'):
    """Evaluate transfer: source classifier → target domain"""
    print(f"\n{'=' * 70}")
    print(f"TRANSFER: {source_name} → {target_name}")
    print(f"{'=' * 70}")

    shared_encoder.eval()
    projection.eval()
    classifier.eval()

    shared_encoder.to(device)
    projection.to(device)
    classifier.to(device)

    correct = total = 0
    class_correct = [0] * 10
    class_total   = [0] * 10

    with torch.no_grad():
        for x, s in test_loader:
            sparse = x['sparse_image'].to(device)
            mask   = x['mask'].to(device)
            labels = s['label'].to(device)

            z      = shared_encoder(sparse, mask, domain_id)
            z      = projection(z)
            logits = classifier(z)

            _, pred = logits.max(1)
            total   += labels.size(0)
            correct += pred.eq(labels).sum().item()

            for i in range(labels.size(0)):
                tl = labels[i].item()
                class_correct[tl] += (pred[i] == tl).item()
                class_total[tl]   += 1

    acc = 100. * correct / total
    print(f"Overall Accuracy: {acc:.2f}%")
    print("Per-Class:")
    for i in range(10):
        if class_total[i] > 0:
            print(f"  Class {i}: {100.*class_correct[i]/class_total[i]:.2f}%"
                  f" ({class_correct[i]}/{class_total[i]})")
    return acc


# ============================================================================
# 14. MAIN EXECUTION
# ============================================================================

if __name__ == '__main__':
    print("\n" + "=" * 70)
    print("THREE-DOMAIN UNIVERSAL TRANSFER LEARNING")
    print("Domains: MNIST + USPS + SVHN (digits 0-9, semantically meaningful)")
    print("Architecture: Shared encoder + Domain-Specific BatchNorm")
    print("Same Z space for all domains — universal manifold by construction")
    print("Nine transfer evaluations (all source→target combinations)")
    print("=" * 70 + "\n")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    SPARSITY        = 0.15
    LATENT_CHANNELS = 128
    BATCH_SIZE      = 64

    pad_and_repeat = PadAndRepeatChannels()

    # ==================== TRANSFORMS ====================

    # MNIST: 28×28×1 → pad to 32×32 → repeat 3ch
    mnist_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
        transforms.Lambda(pad_and_repeat)
    ])

    # USPS: USPSDataset already returns tensor [-1,1]
    # Only apply pad_and_repeat: 16×16×1 → 32×32×3
    usps_transform = transforms.Lambda(pad_and_repeat)

    # SVHN: 32×32×3 already correct
    svhn_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # ==================== LOAD DATASETS ====================
    print("Loading datasets...")

    mnist_train_full = datasets.MNIST(
        './data', train=True, download=True, transform=mnist_transform
    )
    mnist_test_full = datasets.MNIST(
        './data', train=False, transform=mnist_transform
    )

    # USPS: manual download (avoids broken torchvision URL)
    usps_train_full = USPSDataset(
        './data', train=True, transform=usps_transform, download=True
    )
    usps_test_full = USPSDataset(
        './data', train=False, transform=usps_transform, download=True
    )

    svhn_train_full = datasets.SVHN(
        './data', split='train', download=True, transform=svhn_transform
    )
    svhn_test_full = datasets.SVHN(
        './data', split='test', download=True, transform=svhn_transform
    )

    print("✓ All datasets loaded\n")

    # ==================== SPLIT ====================

    def split_dataset(full_dataset, seed=42):
        n = len(full_dataset)
        train_size = int(0.90 * n)
        val_size   = int(0.05 * n)
        idx = list(range(n))
        np.random.seed(seed)
        np.random.shuffle(idx)
        return (Subset(full_dataset, idx[:train_size]),
                Subset(full_dataset, idx[train_size:train_size+val_size]),
                Subset(full_dataset, idx[train_size+val_size:]))

    mnist_train, mnist_val, mnist_test = split_dataset(mnist_train_full)
    usps_train,  usps_val,  usps_test  = split_dataset(usps_train_full)
    svhn_train,  svhn_val,  svhn_test  = split_dataset(svhn_train_full)

    # ==================== SPARSE DATASETS ====================

    mnist_tr = SparseMultiDomainDataset(mnist_train, SPARSITY, DOMAIN_MNIST, 'MNIST-Train')
    mnist_vl = SparseMultiDomainDataset(mnist_val,   SPARSITY, DOMAIN_MNIST, 'MNIST-Val')
    mnist_te = SparseMultiDomainDataset(mnist_test,  SPARSITY, DOMAIN_MNIST, 'MNIST-Test')

    usps_tr  = SparseMultiDomainDataset(usps_train,  SPARSITY, DOMAIN_USPS,  'USPS-Train')
    usps_vl  = SparseMultiDomainDataset(usps_val,    SPARSITY, DOMAIN_USPS,  'USPS-Val')
    usps_te  = SparseMultiDomainDataset(usps_test,   SPARSITY, DOMAIN_USPS,  'USPS-Test')

    svhn_tr  = SparseMultiDomainDataset(svhn_train,  SPARSITY, DOMAIN_SVHN,  'SVHN-Train')
    svhn_vl  = SparseMultiDomainDataset(svhn_val,    SPARSITY, DOMAIN_SVHN,  'SVHN-Val')
    svhn_te  = SparseMultiDomainDataset(svhn_test,   SPARSITY, DOMAIN_SVHN,  'SVHN-Test')

    # ==================== SVHN WEIGHTED SAMPLER ====================

    def get_svhn_weights(dataset):
        labels = [dataset[i][1]['label'] for i in range(len(dataset))]
        labels = np.array(labels)
        counts = np.bincount(labels, minlength=10)
        weights = 1.0 / (counts[labels] + 1e-6)
        return torch.FloatTensor(weights)

    print("Computing SVHN sample weights...")
    svhn_weights = get_svhn_weights(svhn_tr)
    svhn_sampler = WeightedRandomSampler(svhn_weights, len(svhn_weights), replacement=True)
    print("✓ Done\n")

    # ==================== DATALOADERS ====================

    mnist_train_loader = DataLoader(mnist_tr, BATCH_SIZE, shuffle=True,  num_workers=0)
    mnist_val_loader   = DataLoader(mnist_vl, BATCH_SIZE, shuffle=False, num_workers=0)
    mnist_test_loader  = DataLoader(mnist_te, BATCH_SIZE, shuffle=False, num_workers=0)

    usps_train_loader  = DataLoader(usps_tr, BATCH_SIZE, shuffle=True,  num_workers=0)
    usps_val_loader    = DataLoader(usps_vl, BATCH_SIZE, shuffle=False, num_workers=0)
    usps_test_loader   = DataLoader(usps_te, BATCH_SIZE, shuffle=False, num_workers=0)

    svhn_train_loader  = DataLoader(svhn_tr, BATCH_SIZE, sampler=svhn_sampler, num_workers=0)
    svhn_val_loader    = DataLoader(svhn_vl, BATCH_SIZE, shuffle=False, num_workers=0)
    svhn_test_loader   = DataLoader(svhn_te, BATCH_SIZE, shuffle=False, num_workers=0)

    # ==================== BUILD MODELS ====================

    shared_encoder = SharedUniversalEncoder(
        latent_channels=LATENT_CHANNELS, num_res_blocks=3, num_domains=NUM_DOMAINS
    )
    decoder_mnist = DomainSpecificDecoder(LATENT_CHANNELS, num_res_blocks=2)
    decoder_usps  = DomainSpecificDecoder(LATENT_CHANNELS, num_res_blocks=2)
    decoder_svhn  = DomainSpecificDecoder(LATENT_CHANNELS, num_res_blocks=2)

    # ==================== STEP 1 ====================

    print("\n" + "=" * 70)
    print("STEP 1: JOINT PRE-TRAINING — SHARED ENCODER + DOMAIN-SPECIFIC BN")
    print("=" * 70 + "\n")

    shared_encoder, decoder_mnist, decoder_usps, decoder_svhn = \
        pretrain_shared_encoder(
            shared_encoder,
            decoder_mnist, decoder_usps, decoder_svhn,
            mnist_train_loader, usps_train_loader, svhn_train_loader,
            mnist_val_loader,   usps_val_loader,   svhn_val_loader,
            max_epochs=100, patience=15, device=device
        )

    # ==================== STEP 2 ====================

    print("\n" + "=" * 70)
    print("STEP 2: THREE-WAY PROJECTION + ALIGNMENT")
    print("=" * 70 + "\n")

    projection_mnist = DomainInvariantProjection(LATENT_CHANNELS, num_res_blocks=2)
    projection_usps  = DomainInvariantProjection(LATENT_CHANNELS, num_res_blocks=2)
    projection_svhn  = DomainInvariantProjection(LATENT_CHANNELS, num_res_blocks=2)

    projection_mnist, projection_usps, projection_svhn = \
        train_three_way_alignment(
            shared_encoder,
            projection_mnist, projection_usps, projection_svhn,
            decoder_mnist, decoder_usps, decoder_svhn,
            mnist_train_loader, usps_train_loader, svhn_train_loader,
            mnist_val_loader,   usps_val_loader,   svhn_val_loader,
            max_epochs=100, patience=15, device=device
        )

    # ==================== STEP 3: THREE CLASSIFIERS ====================

    classifier_mnist = SpatialClassifier(LATENT_CHANNELS, 10)
    classifier_usps  = SpatialClassifier(LATENT_CHANNELS, 10)
    classifier_svhn  = SpatialClassifier(LATENT_CHANNELS, 10)

    print("\n" + "=" * 70)
    print("STEP 3A: CLASSIFIER ON MNIST ONLY")
    print("=" * 70)
    classifier_mnist = train_classifier_single_domain(
        shared_encoder, projection_mnist, classifier_mnist,
        mnist_train_loader, mnist_val_loader,
        DOMAIN_MNIST, "MNIST", max_epochs=50, patience=10, device=device
    )

    print("\n" + "=" * 70)
    print("STEP 3B: CLASSIFIER ON USPS ONLY")
    print("=" * 70)
    classifier_usps = train_classifier_single_domain(
        shared_encoder, projection_usps, classifier_usps,
        usps_train_loader, usps_val_loader,
        DOMAIN_USPS, "USPS", max_epochs=50, patience=10, device=device
    )

    print("\n" + "=" * 70)
    print("STEP 3C: CLASSIFIER ON SVHN ONLY")
    print("=" * 70)
    classifier_svhn = train_classifier_single_domain(
        shared_encoder, projection_svhn, classifier_svhn,
        svhn_train_loader, svhn_val_loader,
        DOMAIN_SVHN, "SVHN", max_epochs=50, patience=10, device=device
    )

    # ==================== STEP 4: NINE EVALUATIONS ====================

    print("\n" + "=" * 70)
    print("STEP 4: ALL NINE TRANSFER EVALUATIONS")
    print("=" * 70 + "\n")

    # MNIST classifier → all targets
    r_mm = evaluate_transfer(shared_encoder, projection_mnist, classifier_mnist,
                              mnist_test_loader, DOMAIN_MNIST, "MNIST Clf", "MNIST", device)
    r_mu = evaluate_transfer(shared_encoder, projection_usps,  classifier_mnist,
                              usps_test_loader,  DOMAIN_USPS,  "MNIST Clf", "USPS (TRANSFER)", device)
    r_ms = evaluate_transfer(shared_encoder, projection_svhn,  classifier_mnist,
                              svhn_test_loader,  DOMAIN_SVHN,  "MNIST Clf", "SVHN (TRANSFER)", device)

    # USPS classifier → all targets
    r_um = evaluate_transfer(shared_encoder, projection_mnist, classifier_usps,
                              mnist_test_loader, DOMAIN_MNIST, "USPS Clf", "MNIST (TRANSFER)", device)
    r_uu = evaluate_transfer(shared_encoder, projection_usps,  classifier_usps,
                              usps_test_loader,  DOMAIN_USPS,  "USPS Clf", "USPS", device)
    r_us = evaluate_transfer(shared_encoder, projection_svhn,  classifier_usps,
                              svhn_test_loader,  DOMAIN_SVHN,  "USPS Clf", "SVHN (TRANSFER)", device)

    # SVHN classifier → all targets
    r_sm = evaluate_transfer(shared_encoder, projection_mnist, classifier_svhn,
                              mnist_test_loader, DOMAIN_MNIST, "SVHN Clf", "MNIST (TRANSFER)", device)
    r_su = evaluate_transfer(shared_encoder, projection_usps,  classifier_svhn,
                              usps_test_loader,  DOMAIN_USPS,  "SVHN Clf", "USPS (TRANSFER)", device)
    r_ss = evaluate_transfer(shared_encoder, projection_svhn,  classifier_svhn,
                              svhn_test_loader,  DOMAIN_SVHN,  "SVHN Clf", "SVHN", device)

    # ==================== FINAL SUMMARY ====================

    print("\n" + "=" * 70)
    print("FINAL RESULTS — THREE-DOMAIN UNIVERSAL TRANSFER")
    print("All domains share semantic meaning: digits 0-9")
    print("Shared encoder + Domain-Specific BN → same Z space")
    print("=" * 70)
    print(f"""
┌──────────────────┬──────────┬──────────┬──────────┐
│ Classifier →     │  MNIST   │   USPS   │   SVHN   │
│ Target Domain ↓  │          │          │          │
├──────────────────┼──────────┼──────────┼──────────┤
│ MNIST            │ {r_mm:6.2f}%  │ {r_um:6.2f}%  │ {r_sm:6.2f}%  │
│ USPS             │ {r_mu:6.2f}%  │ {r_uu:6.2f}%  │ {r_su:6.2f}%  │
│ SVHN             │ {r_ms:6.2f}%  │ {r_us:6.2f}%  │ {r_ss:6.2f}%  │
└──────────────────┴──────────┴──────────┴──────────┘
Diagonal  = same domain baseline
Off-diag  = zero-shot cross-domain transfer
    """)

    # ==================== SAVE ====================

    results = {
        'architecture': 'shared_encoder_domain_specific_BN',
        'sparsity': SPARSITY,
        'latent_channels': LATENT_CHANNELS,
        'same_domain':   {'mm': r_mm, 'uu': r_uu, 'ss': r_ss},
        'cross_domain':  {
            'mnist_to_usps': r_mu, 'mnist_to_svhn': r_ms,
            'usps_to_mnist': r_um, 'usps_to_svhn':  r_us,
            'svhn_to_mnist': r_sm, 'svhn_to_usps':  r_su,
        }
    }

    with open('three_domain_domainBN_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    torch.save({
        'shared_encoder':   shared_encoder.state_dict(),
        'projection_mnist': projection_mnist.state_dict(),
        'projection_usps':  projection_usps.state_dict(),
        'projection_svhn':  projection_svhn.state_dict(),
    }, 'three_domain_domainBN_manifold.pth')

    torch.save(classifier_mnist.state_dict(), 'three_domain_domainBN_clf_mnist.pth')
    torch.save(classifier_usps.state_dict(),  'three_domain_domainBN_clf_usps.pth')
    torch.save(classifier_svhn.state_dict(),  'three_domain_domainBN_clf_svhn.pth')

    print("✓ All results and models saved!")
    print("  - three_domain_domainBN_results.json")
    print("  - three_domain_domainBN_manifold.pth")
    print("  - three_domain_domainBN_clf_mnist/usps/svhn.pth")
    print("=" * 70)


THREE-DOMAIN UNIVERSAL TRANSFER LEARNING
Domains: MNIST + USPS + SVHN (digits 0-9, semantically meaningful)
Architecture: Shared encoder + Domain-Specific BatchNorm
Same Z space for all domains — universal manifold by construction
Nine transfer evaluations (all source→target combinations)

Using device: cuda

Loading datasets...
✓ All datasets loaded

MNIST-Train Sparse Dataset — Size: 54000, Sparsity: 15% visible
MNIST-Val Sparse Dataset — Size: 3000, Sparsity: 15% visible
MNIST-Test Sparse Dataset — Size: 3000, Sparsity: 15% visible
USPS-Train Sparse Dataset — Size: 6561, Sparsity: 15% visible
USPS-Val Sparse Dataset — Size: 364, Sparsity: 15% visible
USPS-Test Sparse Dataset — Size: 366, Sparsity: 15% visible
SVHN-Train Sparse Dataset — Size: 65931, Sparsity: 15% visible
SVHN-Val Sparse Dataset — Size: 3662, Sparsity: 15% visible
SVHN-Test Sparse Dataset — Size: 3664, Sparsity: 15% visible
Computing SVHN sample weights...
✓ Done


STEP 1: JOINT PRE-TRAINING — SHARED ENCODER + DOMAI

In [2]:
"""
Topological Evaluation of Three-Domain Universal Manifold
==========================================================

Loads: three_domain_domainBN_manifold.pth
       three_domain_domainBN_clf_mnist/usps/svhn.pth

Evaluates the ENTIRE unified Z space as ONE manifold:
  Z_total = [Z_mnist || Z_usps || Z_svhn]

Metrics (all using cosine distance):
  - β₀  : connected components (persistent homology)
  - β₁  : loops/holes         (persistent homology, subsampled)
  - W₂  : sliced Wasserstein-2 on Z_total
  - Trust Score
  - Continuity
  - Alignment Error (cross-domain centroid consistency)

Sparsity: ρ = 0.15 only
Sample:   15% of each domain's full dataset
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset
import numpy as np
import json
import os
import urllib.request
import bz2
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import wasserstein_distance
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

# Try importing ripser for persistent homology
try:
    from ripser import ripser
    RIPSER_AVAILABLE = True
    print("✓ ripser available — β₀ and β₁ will use persistent homology")
except ImportError:
    RIPSER_AVAILABLE = False
    print("⚠ ripser not available — β₀ via graph components, β₁ skipped")
    print("  Install with: pip install ripser")

DOMAIN_MNIST = 0
DOMAIN_USPS  = 1
DOMAIN_SVHN  = 2
NUM_DOMAINS  = 3
SPARSITY     = 0.15
SAMPLE_FRAC  = 0.15  # 15% of each domain


# ============================================================================
# 1. USPS MANUAL DOWNLOAD (same as training script)
# ============================================================================

class USPSDataset(Dataset):
    URLS = {
        'train': 'https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/multiclass/usps.bz2',
        'test':  'https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/multiclass/usps.t.bz2'
    }

    def __init__(self, root, train=True, transform=None, download=True):
        self.root = os.path.join(root, 'usps_manual')
        self.train = train
        self.transform = transform
        os.makedirs(self.root, exist_ok=True)

        split = 'train' if train else 'test'
        filepath = os.path.join(self.root, f'usps_{split}.bz2')

        if download and not os.path.exists(filepath):
            print(f'Downloading USPS {split}...')
            urllib.request.urlretrieve(self.URLS[split], filepath)
            print(f'✓ Downloaded USPS {split}')

        self.data, self.labels = self._load_libsvm(filepath)

    def _load_libsvm(self, filepath):
        data, labels = [], []
        with bz2.open(filepath, 'rt') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                labels.append(int(parts[0]) % 10)
                features = np.zeros(256, dtype=np.float32)
                for feat in parts[1:]:
                    if ':' in feat:
                        idx, val = feat.split(':')
                        features[int(idx) - 1] = float(val)
                img = features.reshape(16, 16)
                img = (img + 1.0) / 2.0
                data.append(img)
        return np.array(data, dtype=np.float32), np.array(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.FloatTensor(self.data[idx]).unsqueeze(0)
        img = img * 2.0 - 1.0
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label


# ============================================================================
# 2. PREPROCESSING (same as training script)
# ============================================================================

class PadAndRepeatChannels:
    def __call__(self, x):
        c, h, w = x.shape
        pad_h  = (32 - h) // 2
        pad_w  = (32 - w) // 2
        pad_h2 = 32 - h - pad_h
        pad_w2 = 32 - w - pad_w
        x = F.pad(x, [pad_w, pad_w2, pad_h, pad_h2], value=0)
        x = x.repeat(3, 1, 1)
        return x


# ============================================================================
# 3. SPARSE DATASET
# ============================================================================

class SparseMultiDomainDataset(Dataset):
    def __init__(self, base_dataset, sparsity_level=0.15,
                 domain_id=0, domain_name='Unknown'):
        self.base_dataset   = base_dataset
        self.sparsity_level = sparsity_level
        self.domain_id      = domain_id
        self.domain_name    = domain_name

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        full_image, label = self.base_dataset[idx]
        spatial_mask = (torch.rand(1, 32, 32) < self.sparsity_level).float()
        mask         = spatial_mask.repeat(3, 1, 1)
        sparse_image = full_image * mask
        label        = int(label) % 10

        return {
            'sparse_image': sparse_image,
            'mask':         mask,
            'full_image':   full_image,
            'label':        label,
            'domain':       self.domain_id
        }


# ============================================================================
# 4. MODEL ARCHITECTURES (must match training script exactly)
# ============================================================================

class ResidualBlockDomainBN(nn.Module):
    def __init__(self, channels, num_domains=NUM_DOMAINS):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.ModuleList([nn.BatchNorm2d(channels) for _ in range(num_domains)])
        self.bn2   = nn.ModuleList([nn.BatchNorm2d(channels) for _ in range(num_domains)])

    def forward(self, x, domain_id):
        residual = x
        out = F.relu(self.bn1[domain_id](x))
        out = self.conv1(out)
        out = F.relu(self.bn2[domain_id](out))
        out = self.conv2(out)
        return out + residual


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(x))
        out = self.conv1(out)
        out = F.relu(self.bn2(out))
        out = self.conv2(out)
        return out + residual


class SharedUniversalEncoder(nn.Module):
    def __init__(self, latent_channels=128, num_res_blocks=3, num_domains=NUM_DOMAINS):
        super().__init__()
        self.latent_channels = latent_channels
        self.num_domains     = num_domains

        self.input_conv = nn.Conv2d(6, 32, 3, padding=1, bias=False)
        self.input_bn   = nn.ModuleList([nn.BatchNorm2d(32) for _ in range(num_domains)])

        self.res1  = nn.ModuleList([ResidualBlockDomainBN(32,  num_domains) for _ in range(num_res_blocks)])
        self.down1 = nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False)
        self.down1_bn = nn.ModuleList([nn.BatchNorm2d(64) for _ in range(num_domains)])

        self.res2  = nn.ModuleList([ResidualBlockDomainBN(64,  num_domains) for _ in range(num_res_blocks)])
        self.down2 = nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False)
        self.down2_bn = nn.ModuleList([nn.BatchNorm2d(128) for _ in range(num_domains)])

        self.res3  = nn.ModuleList([ResidualBlockDomainBN(128, num_domains) for _ in range(num_res_blocks)])
        self.down3 = nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False)
        self.down3_bn = nn.ModuleList([nn.BatchNorm2d(256) for _ in range(num_domains)])

        self.res4  = nn.ModuleList([ResidualBlockDomainBN(256, num_domains) for _ in range(num_res_blocks)])

        self.bottleneck    = nn.Conv2d(256, latent_channels, 1, bias=False)
        self.bottleneck_bn = nn.ModuleList([nn.BatchNorm2d(latent_channels) for _ in range(num_domains)])

        self.res_bottleneck = nn.ModuleList([
            ResidualBlockDomainBN(latent_channels, num_domains)
            for _ in range(num_res_blocks)
        ])

    def forward(self, sparse_image, mask, domain_id):
        x = torch.cat([sparse_image, mask], dim=1)
        x = F.relu(self.input_bn[domain_id](self.input_conv(x)))
        for block in self.res1:
            x = block(x, domain_id)
        x = F.relu(self.down1_bn[domain_id](self.down1(x)))
        for block in self.res2:
            x = block(x, domain_id)
        x = F.relu(self.down2_bn[domain_id](self.down2(x)))
        for block in self.res3:
            x = block(x, domain_id)
        x = F.relu(self.down3_bn[domain_id](self.down3(x)))
        for block in self.res4:
            x = block(x, domain_id)
        x = F.relu(self.bottleneck_bn[domain_id](self.bottleneck(x)))
        for block in self.res_bottleneck:
            x = block(x, domain_id)
        return x


class DomainInvariantProjection(nn.Module):
    def __init__(self, latent_channels=128, num_res_blocks=2):
        super().__init__()
        self.projection = nn.Sequential(
            *[ResidualBlock(latent_channels) for _ in range(num_res_blocks)],
            nn.Conv2d(latent_channels, latent_channels, 1),
            nn.BatchNorm2d(latent_channels)
        )

    def forward(self, z):
        return self.projection(z)


# ============================================================================
# 5. ENCODE ONE DOMAIN → FLAT LATENT CODES
# ============================================================================

def encode_domain(shared_encoder, projection, dataset,
                  domain_id, device='cpu', batch_size=64):
    """
    Encode all samples in dataset through shared encoder + projection.
    Returns flat latent codes [N, D] using cosine-friendly L2 normalization.
    """
    shared_encoder.eval()
    projection.eval()

    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=0)

    all_z      = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            sparse = batch['sparse_image'].to(device)
            mask   = batch['mask'].to(device)
            labels = batch['label']

            z = shared_encoder(sparse, mask, domain_id)
            z = projection(z)

            # Flatten spatial dims: [B, C, H, W] → [B, C*H*W]
            z_flat = z.view(z.size(0), -1)

            # L2 normalize for cosine distance consistency
            z_flat = F.normalize(z_flat, p=2, dim=1)

            all_z.append(z_flat.cpu().numpy())
            all_labels.extend([int(l) % 10 for l in labels])

    return np.vstack(all_z), np.array(all_labels)


# ============================================================================
# 6. TOPOLOGICAL METRICS
# ============================================================================

def compute_betti_numbers(latent_codes, subsample=2000, max_dim=1):
    """
    Compute β₀ and β₁ using persistent homology (ripser).

    β₀ = number of connected components (should be 1)
    β₁ = number of loops/holes (reveals topological shape)

    Uses cosine distance.
    Subsamples to `subsample` points for efficiency.
    """
    if not RIPSER_AVAILABLE:
        # Fallback: β₀ via graph components, β₁ = N/A
        beta_0 = compute_betti_0_graph(latent_codes)
        return beta_0, None

    N = len(latent_codes)
    if N > subsample:
        idx = np.random.choice(N, subsample, replace=False)
        codes = latent_codes[idx]
    else:
        codes = latent_codes

    print(f"  Computing persistent homology on {len(codes)} points...")

    # Cosine distance matrix
    dist_matrix = cosine_distances(codes).astype(np.float32)

    # Run ripser up to dim 1 (β₀ and β₁)
    result = ripser(dist_matrix, maxdim=max_dim, distance_matrix=True)
    diagrams = result['dgms']

    # β₀: count components that persist (not born and die at 0)
    # Components that never die have death = inf
    h0 = diagrams[0]
    beta_0 = int(np.sum(h0[:, 1] == np.inf))  # immortal components

    # β₁: count loops with significant persistence
    if len(diagrams) > 1:
        h1 = diagrams[1]
        # Count loops with persistence > threshold (filter noise)
        if len(h1) > 0:
            persistence = h1[:, 1] - h1[:, 0]
            threshold = np.percentile(persistence, 75)  # top 25% persist
            beta_1 = int(np.sum(persistence > threshold))
        else:
            beta_1 = 0
    else:
        beta_1 = 0

    return beta_0, beta_1


def compute_betti_0_graph(latent_codes, kappa=10):
    """
    Fallback β₀ computation via kNN graph connected components.
    Uses cosine distance.
    """
    N = len(latent_codes)
    nn_model = NearestNeighbors(
        n_neighbors=kappa+1, metric='cosine',
        algorithm='brute', n_jobs=-1
    )
    nn_model.fit(latent_codes)
    _, neighbors = nn_model.kneighbors(latent_codes)

    rows, cols = [], []
    for i in range(N):
        for j in neighbors[i, 1:]:
            rows.extend([i, j])
            cols.extend([j, i])

    data = np.ones(len(rows))
    adj  = csr_matrix((data, (rows, cols)), shape=(N, N))
    n_components, _ = connected_components(adj, directed=False)
    return int(n_components)


def compute_trust_score(latent_codes, labels, kappa=5):
    """
    Trust Score using cosine distance.
    Measures: do k-nearest neighbors share the same class label?
    High trust → semantic neighborhoods preserved in Z space.
    """
    N = len(latent_codes)
    nn_model = NearestNeighbors(
        n_neighbors=kappa+1, metric='cosine',
        algorithm='brute', n_jobs=-1
    )
    nn_model.fit(latent_codes)
    _, neighbors = nn_model.kneighbors(latent_codes)
    neighbors = neighbors[:, 1:]  # exclude self

    trust_scores = []
    for i in range(N):
        nb_labels = labels[neighbors[i]]
        trust_scores.append(np.mean(nb_labels == labels[i]))

    return float(np.mean(trust_scores))


def compute_sliced_wasserstein_cosine(X, n_projections=200):
    """
    Sliced Wasserstein-2 distance on the ENTIRE Z_total distribution.
    Measures how compact and well-distributed the unified manifold is.

    We project onto random unit vectors and compute the
    self-Wasserstein distance (how uniform the projected distribution is).
    Lower = more uniform = better manifold coverage.

    Uses cosine-normalized embeddings (already L2 normalized).
    """
    D = X.shape[1]
    distances = []

    # Split into two halves — measure how consistent the distribution is
    half = len(X) // 2
    X1 = X[:half]
    X2 = X[half:2*half]

    for _ in range(n_projections):
        theta  = np.random.randn(D)
        theta  = theta / (np.linalg.norm(theta) + 1e-8)
        p1     = X1 @ theta
        p2     = X2 @ theta
        w1d    = wasserstein_distance(p1, p2)
        distances.append(w1d ** 2)

    return float(np.sqrt(np.mean(distances)))


def compute_continuity(latent_codes, labels, kappa=5):
    """
    Continuity using cosine distance.
    Measures: are cosine neighbors in Z also neighbors in label space?
    High continuity → no spurious proximities introduced by encoding.
    """
    N = len(latent_codes)
    nn_model = NearestNeighbors(
        n_neighbors=kappa+1, metric='cosine',
        algorithm='brute', n_jobs=-1
    )
    nn_model.fit(latent_codes)
    _, neighbors = nn_model.kneighbors(latent_codes)
    neighbors = neighbors[:, 1:]

    # For each point: what fraction of its Z-neighbors share its label?
    cont_scores = []
    for i in range(N):
        nb_labels = labels[neighbors[i]]
        cont_scores.append(np.mean(nb_labels == labels[i]))

    return float(np.mean(cont_scores))


def compute_alignment_error(z_mnist, z_usps, z_svhn,
                             labels_mnist, labels_usps, labels_svhn):
    """
    Cross-domain alignment error using cosine distance.

    For each class c, compute centroid in each domain:
        μ_c^mnist, μ_c^usps, μ_c^svhn

    Alignment error = mean cosine distance between centroids across all
    domain pairs and all classes.

    Low alignment → all three domains' class centroids are geometrically
    coincident in Z space → verified homeomorphic structure.
    """
    num_classes = 10
    errors = []

    for c in range(num_classes):
        mask_m = (labels_mnist == c)
        mask_u = (labels_usps  == c)
        mask_s = (labels_svhn  == c)

        if mask_m.sum() == 0 or mask_u.sum() == 0 or mask_s.sum() == 0:
            continue

        cent_m = z_mnist[mask_m].mean(axis=0, keepdims=True)
        cent_u = z_usps[mask_u].mean(axis=0, keepdims=True)
        cent_s = z_svhn[mask_s].mean(axis=0, keepdims=True)

        # Cosine distances between centroids
        d_mu = float(cosine_distances(cent_m, cent_u)[0, 0])
        d_ms = float(cosine_distances(cent_m, cent_s)[0, 0])
        d_us = float(cosine_distances(cent_u, cent_s)[0, 0])

        errors.append((d_mu + d_ms + d_us) / 3.0)

    return float(np.mean(errors))


# ============================================================================
# 7. MAIN EVALUATION
# ============================================================================

if __name__ == '__main__':
    print("\n" + "=" * 70)
    print("TOPOLOGICAL EVALUATION — THREE-DOMAIN UNIVERSAL MANIFOLD")
    print("Z_total = [Z_mnist || Z_usps || Z_svhn] as ONE unified space")
    print(f"Sparsity: ρ = {SPARSITY}, Sample: {SAMPLE_FRAC*100:.0f}% per domain")
    print("All metrics use cosine distance")
    print("=" * 70 + "\n")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")

    LATENT_CHANNELS = 128

    # ==================== LOAD DATASETS ====================

    pad_and_repeat = PadAndRepeatChannels()

    mnist_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
        transforms.Lambda(pad_and_repeat)
    ])
    usps_transform = transforms.Lambda(pad_and_repeat)
    svhn_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    print("Loading datasets...")

    mnist_full = torch.utils.data.ConcatDataset([
        datasets.MNIST('./data', train=True,  download=True, transform=mnist_transform),
        datasets.MNIST('./data', train=False, download=True, transform=mnist_transform)
    ])
    usps_full = torch.utils.data.ConcatDataset([
        USPSDataset('./data', train=True,  transform=usps_transform, download=True),
        USPSDataset('./data', train=False, transform=usps_transform, download=True)
    ])
    svhn_full = torch.utils.data.ConcatDataset([
        datasets.SVHN('./data', split='train', download=True, transform=svhn_transform),
        datasets.SVHN('./data', split='test',  download=True, transform=svhn_transform)
    ])

    # 15% random subsample per domain
    def subsample(dataset, frac=SAMPLE_FRAC, seed=42):
        n = len(dataset)
        k = max(1, int(n * frac))
        np.random.seed(seed)
        idx = np.random.choice(n, k, replace=False)
        return Subset(dataset, idx)

    mnist_sub = subsample(mnist_full)
    usps_sub  = subsample(usps_full)
    svhn_sub  = subsample(svhn_full)

    print(f"  MNIST:  {len(mnist_sub)} samples ({SAMPLE_FRAC*100:.0f}% of {len(mnist_full)})")
    print(f"  USPS:   {len(usps_sub)} samples ({SAMPLE_FRAC*100:.0f}% of {len(usps_full)})")
    print(f"  SVHN:   {len(svhn_sub)} samples ({SAMPLE_FRAC*100:.0f}% of {len(svhn_full)})")

    # Wrap in sparse dataset
    mnist_sparse = SparseMultiDomainDataset(mnist_sub, SPARSITY, DOMAIN_MNIST, 'MNIST')
    usps_sparse  = SparseMultiDomainDataset(usps_sub,  SPARSITY, DOMAIN_USPS,  'USPS')
    svhn_sparse  = SparseMultiDomainDataset(svhn_sub,  SPARSITY, DOMAIN_SVHN,  'SVHN')

    print("✓ Datasets ready\n")

    # ==================== LOAD MODELS ====================

    print("Loading three_domain_domainBN_manifold.pth...")

    shared_encoder   = SharedUniversalEncoder(LATENT_CHANNELS, num_res_blocks=3)
    projection_mnist = DomainInvariantProjection(LATENT_CHANNELS, num_res_blocks=2)
    projection_usps  = DomainInvariantProjection(LATENT_CHANNELS, num_res_blocks=2)
    projection_svhn  = DomainInvariantProjection(LATENT_CHANNELS, num_res_blocks=2)

    ckpt = torch.load('three_domain_domainBN_manifold.pth', map_location=device)
    shared_encoder.load_state_dict(ckpt['shared_encoder'])
    projection_mnist.load_state_dict(ckpt['projection_mnist'])
    projection_usps.load_state_dict(ckpt['projection_usps'])
    projection_svhn.load_state_dict(ckpt['projection_svhn'])

    shared_encoder.to(device)
    projection_mnist.to(device)
    projection_usps.to(device)
    projection_svhn.to(device)

    print("✓ Models loaded\n")

    # ==================== ENCODE ALL THREE DOMAINS ====================

    print("Encoding Z_mnist...")
    z_mnist, labels_mnist = encode_domain(
        shared_encoder, projection_mnist,
        mnist_sparse, DOMAIN_MNIST, device
    )
    print(f"  Z_mnist shape: {z_mnist.shape}")

    print("Encoding Z_usps...")
    z_usps, labels_usps = encode_domain(
        shared_encoder, projection_usps,
        usps_sparse, DOMAIN_USPS, device
    )
    print(f"  Z_usps shape: {z_usps.shape}")

    print("Encoding Z_svhn...")
    z_svhn, labels_svhn = encode_domain(
        shared_encoder, projection_svhn,
        svhn_sparse, DOMAIN_SVHN, device
    )
    print(f"  Z_svhn shape: {z_svhn.shape}")

    # ==================== COMBINE INTO Z_TOTAL ====================

    Z_total      = np.vstack([z_mnist, z_usps, z_svhn])
    labels_total = np.concatenate([labels_mnist, labels_usps, labels_svhn])
    domains_total = np.array(
        [DOMAIN_MNIST] * len(z_mnist) +
        [DOMAIN_USPS]  * len(z_usps)  +
        [DOMAIN_SVHN]  * len(z_svhn)
    )

    print(f"\nZ_total shape: {Z_total.shape}")
    print(f"  MNIST:  {len(z_mnist)} samples")
    print(f"  USPS:   {len(z_usps)} samples")
    print(f"  SVHN:   {len(z_svhn)} samples")
    print(f"  Total:  {len(Z_total)} samples\n")

    # ==================== COMPUTE METRICS ====================

    results = {}
    kappa = 5

    print("=" * 70)
    print("COMPUTING TOPOLOGICAL METRICS ON Z_total")
    print("=" * 70)

    # --- β₀ and β₁ ---
    print("\n[1/5] Computing β₀ and β₁ (persistent homology)...")
    beta_0, beta_1 = compute_betti_numbers(Z_total, subsample=2000, max_dim=1)
    results['beta_0'] = beta_0
    results['beta_1'] = beta_1 if beta_1 is not None else 'N/A'
    print(f"  β₀ = {beta_0}  (should be 1 — one connected manifold)")
    if beta_1 is not None:
        print(f"  β₁ = {beta_1}  (number of loops in manifold topology)")
    else:
        print(f"  β₁ = N/A (install ripser for β₁)")

    # --- Trust Score ---
    print(f"\n[2/5] Computing Trust Score (κ={kappa}, cosine)...")
    trust = compute_trust_score(Z_total, labels_total, kappa=kappa)
    results['trust_score'] = trust
    status = "✓ PASS" if trust >= 0.80 else "✗ FAIL"
    print(f"  Trust = {trust:.4f}  (threshold ≥ 0.80) {status}")

    # --- Sliced Wasserstein ---
    print("\n[3/5] Computing Sliced Wasserstein-2 on Z_total (cosine)...")
    # Subsample for efficiency
    max_w2 = min(10000, len(Z_total))
    idx_w2 = np.random.choice(len(Z_total), max_w2, replace=False)
    w2 = compute_sliced_wasserstein_cosine(Z_total[idx_w2], n_projections=200)
    results['sliced_w2'] = w2
    status = "✓ PASS" if w2 <= 0.30 else "✗ FAIL"
    print(f"  W₂ = {w2:.4f}  (threshold ≤ 0.30) {status}")

    # --- Continuity ---
    print(f"\n[4/5] Computing Continuity (κ={kappa}, cosine)...")
    cont = compute_continuity(Z_total, labels_total, kappa=kappa)
    results['continuity'] = cont
    status = "✓ PASS" if cont >= 0.70 else "✗ FAIL"
    print(f"  Continuity = {cont:.4f}  (threshold ≥ 0.70) {status}")

    # --- Alignment Error ---
    print("\n[5/5] Computing Cross-Domain Alignment Error (cosine)...")
    align = compute_alignment_error(
        z_mnist, z_usps, z_svhn,
        labels_mnist, labels_usps, labels_svhn
    )
    results['alignment_error'] = align
    status = "✓ PASS" if align <= 0.30 else "✗ FAIL"
    print(f"  Alignment = {align:.4f}  (threshold ≤ 0.30) {status}")

    # ==================== SUMMARY ====================

    print("\n" + "=" * 70)
    print("FINAL RESULTS — THREE-DOMAIN UNIVERSAL MANIFOLD")
    print(f"Z_total = [Z_mnist || Z_usps || Z_svhn], ρ={SPARSITY}")
    print("=" * 70)

    print(f"""
┌─────────────────────────────┬──────────┬───────────┬────────┐
│ Metric                      │  Value   │ Threshold │  Pass  │
├─────────────────────────────┼──────────┼───────────┼────────┤
│ β₀ (connected components)   │ {results['beta_0']:^8} │   = 1     │  {'✓' if results['beta_0']==1 else '✗'}     │
│ β₁ (loops/holes)            │ {str(results['beta_1']):^8} │    —      │  —     │
│ Trust Score τ_t             │ {results['trust_score']:^8.4f} │  ≥ 0.80   │  {'✓' if results['trust_score']>=0.80 else '✗'}     │
│ Sliced W₂ τ_w               │ {results['sliced_w2']:^8.4f} │  ≤ 0.30   │  {'✓' if results['sliced_w2']<=0.30 else '✗'}     │
│ Continuity τ_c              │ {results['continuity']:^8.4f} │  ≥ 0.70   │  {'✓' if results['continuity']>=0.70 else '✗'}     │
│ Alignment Error τ_a         │ {results['alignment_error']:^8.4f} │  ≤ 0.30   │  {'✓' if results['alignment_error']<=0.30 else '✗'}     │
└─────────────────────────────┴──────────┴───────────┴────────┘
    """)

    # ==================== LATEX TABLE ====================

    print("=" * 70)
    print("LaTeX Table:")
    print("=" * 70)
    print(r"\begin{table}[!htpb]")
    print(r"\centering")
    print(r"\caption{Topological Unification Verification — Three-Domain Universal Manifold")
    print(r"         ($\kappa=5$, Cosine distance, $\rho=0.15$)}")
    print(r"\label{tab:three_domain_topology}")
    print(r"\begin{tabular}{lcc}")
    print(r"\hline")
    print(r"\textbf{Metric} & \textbf{Value} & \textbf{Threshold} \\")
    print(r"\hline")
    print(f"$\\beta_0$ (connected components) & {results['beta_0']} & $= 1$ \\\\")
    if results['beta_1'] != 'N/A':
        print(f"$\\beta_1$ (loops/holes) & {results['beta_1']} & — \\\\")
    print(f"Trust $\\tau_t$ & {results['trust_score']:.4f} & $\\geq 0.80$ \\\\")
    print(f"Sliced $W_2$ $\\tau_w$ & {results['sliced_w2']:.4f} & $\\leq 0.30$ \\\\")
    print(f"Continuity $\\tau_c$ & {results['continuity']:.4f} & $\\geq 0.70$ \\\\")
    print(f"Alignment $\\tau_a$ & {results['alignment_error']:.4f} & $\\leq 0.30$ \\\\")
    print(r"\hline")
    print(r"\end{tabular}")
    print(r"\end{table}")
    print("=" * 70)

    # ==================== SAVE ====================

    results['sparsity']     = SPARSITY
    results['sample_frac']  = SAMPLE_FRAC
    results['n_mnist']      = len(z_mnist)
    results['n_usps']       = len(z_usps)
    results['n_svhn']       = len(z_svhn)
    results['n_total']      = len(Z_total)
    results['latent_dim']   = Z_total.shape[1]
    results['distance']     = 'cosine'

    with open('three_domain_topology_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("\n✓ Results saved to: three_domain_topology_results.json")
    print("=" * 70)

✓ ripser available — β₀ and β₁ will use persistent homology

TOPOLOGICAL EVALUATION — THREE-DOMAIN UNIVERSAL MANIFOLD
Z_total = [Z_mnist || Z_usps || Z_svhn] as ONE unified space
Sparsity: ρ = 0.15, Sample: 15% per domain
All metrics use cosine distance

Device: cuda

Loading datasets...
  MNIST:  10500 samples (15% of 70000)
  USPS:   1394 samples (15% of 9298)
  SVHN:   14893 samples (15% of 99289)
✓ Datasets ready

Loading three_domain_domainBN_manifold.pth...
✓ Models loaded

Encoding Z_mnist...
  Z_mnist shape: (10500, 2048)
Encoding Z_usps...
  Z_usps shape: (1394, 2048)
Encoding Z_svhn...
  Z_svhn shape: (14893, 2048)

Z_total shape: (26787, 2048)
  MNIST:  10500 samples
  USPS:   1394 samples
  SVHN:   14893 samples
  Total:  26787 samples

COMPUTING TOPOLOGICAL METRICS ON Z_total

[1/5] Computing β₀ and β₁ (persistent homology)...
  Computing persistent homology on 2000 points...
  β₀ = 1  (should be 1 — one connected manifold)
  β₁ = 153  (number of loops in manifold topology